# Task 5: Account security monitoring (unsupervised anomaly detection, F3)

**Goal.** Flag anomalous accounts without labels (only `test.csv` exists). Metric: F3 of class 1, `F3 = 10 TP / (9 T + n)` with T the number of true anomalies and n the number of flagged rows. A flag therefore helps if and only if its probability of being an anomaly exceeds F3/10 (about 9%), so flagging should be generous.

**Final method (as submitted to Kaggle).** The anomalies form separated groups; every step is unsupervised and uses only the Task 5 feature file:
1. **Group A (period-4 change).** Per row, the sum of capped squared robust z-scores of (period 4 - mean of periods 1-3) over all base features; the sorted scores have a clean gap, rows above it are flagged.
2. **Persistent low groups.** Columns with an internal empty gap (small side below 700 rows, on the low side) are grouped into families; a row belongs to a family if it is on the small side in at least two periods (static columns: one).
3. **Missing `behavior_stability`.** Such rows are invisible to step 2; a gradient-boosting classifier trained to recognise the low-stability group from its side effects scores them, and rows with P > 0.09 are flagged.
4. **Two tail rows** with a very large location jump and onset score.

**Result.** 1,857 flags (7.2%). Output: `../outputs/task5_pred.npy`.

In [1]:
import os, re, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

# ---- configuration -------------------------------------------------------------------------------
# DATA = folder of the Kaggle download that holds 01_binary_classification/ ... 05_anomaly_detection/ and sample_submission.csv
DATA = Path(os.environ.get("CPE342_DATA", "")).expanduser()
assert (DATA / "01_binary_classification").exists(), "Set the environment variable CPE342_DATA to the Kaggle download folder (see ../README.md)"
OUT = Path(os.environ.get("CPE342_OUT", "../outputs")); OUT.mkdir(parents=True, exist_ok=True)   # shared by all notebooks
SEED = 342                                   # every random split / model seed in the project
NJOBS = int(os.environ.get("NJOBS", 16))     # LightGBM threads
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

## 1. Data

In [2]:
te = pd.read_csv(DATA / "05_anomaly_detection" / "test.csv"); X = te.drop(columns=["player_id"]); n = len(X)
bases = sorted({re.sub(r"_[1-4]$", "", c) for c in X.columns if re.search(r"_[1-4]$", c)})     # 4 time periods per behaviour column
print(f"{n:,} players x {X.shape[1]} features | {len(bases)} behaviour columns x 4 periods + {X.shape[1] - 4 * len(bases)} static columns")
def rz(s):                                                                                      # robust z-score
    med, iqr = s.median(), s.quantile(.75) - s.quantile(.25); return (s - med) / (iqr / 1.349 if iqr > 0 else (s.std() or 1))

25,889 players x 122 features | 29 behaviour columns x 4 periods + 6 static columns


## 2. Group A: large change in period 4

In [3]:
d4 = pd.DataFrame({b: X[f"{b}_4"] - X[[f"{b}_1", f"{b}_2", f"{b}_3"]].mean(axis=1) for b in bases}).apply(rz)
S = ((d4.clip(-8, 8) ** 2).sum(axis=1, min_count=1) / d4.notna().sum(axis=1) * d4.shape[1]).fillna(0)
v = np.sort(S.values); hi = v[v > 60]; gaps = np.diff(np.log10(hi)); k = int(np.argmax(gaps)); cut = float(np.sqrt(hi[k] * hi[k + 1]))
A = (S > cut).values
print(f"largest gap in the sorted change score above 60: {hi[k]:.1f} -> {hi[k+1]:.1f}; threshold {cut:.0f}; group A = {int(A.sum())} rows ({A.mean():.2%})")
np.save(OUT / "task5_sorted_S.npy", v)

largest gap in the sorted change score above 60: 149.6 -> 221.1; threshold 182; group A = 825 rows (3.19%)


## 3. Persistent low groups (gap scan outside group A)

In [4]:
def gap(v, bins=80):                       # widest empty stretch inside the histogram of a column
    hh, ee = np.histogram(v, bins=bins); nz = np.where(hh > 0)[0]
    g = [(i1 - i0 - 1, (ee[i0 + 1] + ee[i1]) / 2) for i0, i1 in zip(nz[:-1], nz[1:])]
    if not g: return 0, None, 0, None
    ne, c = max(g); low = (v < c).sum() < (v > c).sum(); return ne, c, int(min((v < c).sum(), (v > c).sum())), "low" if low else "high"
rows = []
for c in X.columns:
    vv = X.loc[~A, c].dropna().values
    if len(np.unique(vv)) < 25: continue                  # skip discrete columns
    ne, cc, sm, side = gap(vv)
    if ne >= 4 and sm >= 30: rows.append((c, ne, round(cc, 3), sm, side))
R = pd.DataFrame(rows, columns=["column", "empty_bins", "cut", "n_small_side", "side"]).sort_values("empty_bins", ascending=False)
print(f"{len(R)} columns have an internal gap (>= 4 empty bins of 80, small side >= 30 rows):"); print(R.head(12).to_string(index=False))
R = R[(R.n_small_side <= 700) & (R.side == "low")]         # low-value groups only (drops the discrete zero-mass column and the high sides)
fam = {}
for _, r in R.iterrows(): fam.setdefault(re.sub(r"_[1-4]$", "", r.column), []).append(r.column)
print("families with a gapped low group:", {b: len(c) for b, c in fam.items()})
def odd(col):                                              # rows on the small side of this column's own gap
    v = X.loc[~A, col].dropna().values; hh, ee = np.histogram(v, bins=80); nz = np.where(hh > 0)[0]
    ne, cut_ = max(((i1 - i0 - 1, (ee[i0 + 1] + ee[i1]) / 2) for i0, i1 in zip(nz[:-1], nz[1:])), key=lambda t: t[0])
    return (X[col] < cut_).fillna(False).values
G = {}
for b in fam:
    allcols = [f"{b}_{p}" for p in (1, 2, 3, 4)] if f"{b}_1" in X.columns else [b]
    G[b] = sum(odd(c).astype(int) for c in allcols) >= (2 if len(allcols) > 1 else 1)       # odd in >= 2 periods (static columns: 1)
for b, m in G.items(): print(f"  group {b:24s}: {int(m.sum()):4d} rows | overlap with A {int((m & A).sum())}")
names = list(G); M = np.column_stack([G[b] for b in names] + [A])
print("rows in more than one group:", int((M.sum(1) > 1).sum()))
flags_groups = M.any(1)
print(f"flagged by the groups: {int(flags_groups.sum())} ({flags_groups.mean():.2%})")

21 columns have an internal gap (>= 4 empty bins of 80, small side >= 30 rows):


                   column  empty_bins   cut  n_small_side side
       behavior_stability          31 0.552           246  low
         friend_removes_4          12 0.420          4804  low
    login_time_variance_4          10 1.123           195  low
        ip_hash_entropy_2           9 0.246           199  low
        ip_hash_entropy_1           9 0.247           198  low
        ip_hash_entropy_3           9 0.246           197  low
        ip_hash_entropy_4           9 0.245           194  low
         friend_removes_3           9 0.357          4800  low
communication_frequency_4           8 0.172           249  low
    login_time_variance_2           8 1.047           198  low
         friend_removes_2           7 0.315          4808  low
    login_time_variance_3           7 1.024           199  low
families with a gapped low group: {'behavior_stability': 1, 'login_time_variance': 4, 'ip_hash_entropy': 4, 'communication_frequency': 4, 'session_duration_std': 2, 'activity_level'

## 4. Rows with a missing `behavior_stability`
The low-stability group has side effects (more purchases, fewer logins, older accounts). A classifier trained on the rows with a known stability value recognises the group from those side effects; rows with a missing value are then scored. The threshold 0.09 follows from F3/10.

In [5]:
Mdf = pd.DataFrame(M, columns=names + ["A"]); other = Mdf.drop(columns="behavior_stability").any(axis=1).values
bs = X.behavior_stability.values; grp = np.nan_to_num(bs, nan=9) < 0.552          # 0.552 = the gap cut of behavior_stability found in step 3
print("rule 'behavior_stability < 0.552' reproduces the gap group:", bool((grp == Mdf.behavior_stability.values).all()))
nanb = np.isnan(bs)
Z = X[[c for c in X.columns if c != "behavior_stability"]].copy()
Z["pf_mean"] = X[[f"purchase_frequency_{i}" for i in range(1, 5)]].mean(1); Z["lc_mean"] = X[[f"login_count_{i}" for i in range(1, 5)]].mean(1)
trn = (~nanb) & (~other); yy = grp[trn]; Zt = Z[trn].values
oof = np.zeros(trn.sum())
for a, b in StratifiedKFold(5, shuffle=True, random_state=0).split(Zt, yy):
    oof[b] = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=3, random_state=0).fit(Zt[a], yy[a]).predict_proba(Zt[b])[:, 1]
print(f"OOF AUC of the side-effect classifier {roc_auc_score(yy, oof):.4f} (positives {int(yy.sum())})")
clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=3, random_state=0).fit(Zt, yy)
cand = nanb & (~other); p_nan = np.zeros(n); p_nan[cand] = clf.predict_proba(Z[cand].values)[:, 1]
print(f"{int(cand.sum())} candidate rows with a missing stability value; expected group members about {cand.sum() * yy.mean():.1f}")

rule 'behavior_stability < 0.552' reproduces the gap group: True


OOF AUC of the side-effect classifier 0.9823 (positives 246)


1168 candidate rows with a missing stability value; expected group members about 12.4


## 5. Two tail rows and the final flags
Rows not yet flagged whose login location jumps by more than 20 degrees between two periods **and** whose onset score (periods 3-4 vs periods 1-2) exceeds 200 are far outside the smooth tail of all other rows; exactly two rows qualify.

In [6]:
d34 = pd.DataFrame({b: (X[f"{b}_3"] + X[f"{b}_4"]) / 2 - (X[f"{b}_1"] + X[f"{b}_2"]) / 2 for b in bases}).apply(rz)
S34 = ((d34.clip(-8, 8) ** 2).sum(1, min_count=1) / d34.notna().sum(1) * d34.shape[1]).fillna(0).values
lat = X[[f"login_lat_{p}" for p in (1, 2, 3, 4)]].values; lon = X[[f"login_lon_{p}" for p in (1, 2, 3, 4)]].values
step = np.nan_to_num(np.column_stack([np.hypot(lat[:, t + 1] - lat[:, t], lon[:, t + 1] - lon[:, t]) for t in range(3)]), nan=0).max(1)
tail = np.where(~flags_groups & (step > 20) & (S34 > 200))[0]
print("tail rows (0-based index):", tail.tolist(), "| onset score", np.round(S34[tail], 1).tolist(), "| max location step (deg)", np.round(step[tail], 1).tolist())
assert len(tail) == 2
flags = flags_groups.copy(); flags[tail] = True
nan_add = (p_nan > 0.09) & (~flags)
flags_nan_only = flags_groups | (p_nan > 0.09)           # flags without the two tail rows (used by the alternative final file V53c)
flags = flags | nan_add
print(f"group flags {int(flags_groups.sum())} + tail rows {len(tail)} + missing-stability rows {int(nan_add.sum())} = {int(flags.sum())} flagged ({flags.mean():.2%})")
np.save(OUT / "task5_pred.npy", flags.astype(int)); np.save(OUT / "task5_pred_without_tail.npy", flags_nan_only.astype(int))

tail rows (0-based index): [8038, 14925] | onset score [243.4, 237.2] | max location step (deg) [34.4, 57.0]
group flags 1806 + tail rows 2 + missing-stability rows 49 = 1857 flagged (7.17%)
